In [1]:
!pip install osmnx
!pip install networkx
!pip install folium

  Using cached pandas-2.3.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (91 kB)
  Using cached shapely-2.1.1-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 1.8 MB/s eta 0:00:0000:0100:01
Using cached pandas-2.3.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 1.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 1.7 MB/s eta 0:00:00a 0:00:01m
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached shapely-2.1.1-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━

In [2]:
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import MeasureControl
from IPython.display import display

# Step 1: Download the road network (Antananarivo example)
place_name = "Antananarivo, Madagascar"
G = ox.graph_from_place(place_name, network_type="drive")
G = ox.project_graph(G)

# Step 2: Convert graph to a folium map
center_latlon = ox.geocode(place_name)
m = folium.Map(location=center_latlon, zoom_start=14)
folium.TileLayer('cartodbpositron').add_to(m)

# Add instruction
instructions = folium.Marker(
    location=center_latlon,
    popup="Click on two points to draw a route",
    icon=folium.Icon(color='blue')
)
instructions.add_to(m)

# Add measurement tool
m.add_child(MeasureControl())

# Step 3: Capture clicks (this is limited in Jupyter, use Streamlit for real interactivity)
clicks = []

def on_click(e):
    if len(clicks) < 2:
        clicks.append((e.latlng[0], e.latlng[1]))
        folium.Marker(location=e.latlng).add_to(m)
        if len(clicks) == 2:
            draw_route(clicks[0], clicks[1])

def draw_route(point1, point2):
    # Get nearest nodes
    orig_node = ox.nearest_nodes(G, X=point1[1], Y=point1[0])
    dest_node = ox.nearest_nodes(G, X=point2[1], Y=point2[0])

    # Get shortest path
    route = nx.shortest_path(G, orig_node, dest_node, weight='length')
    route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]

    # Add polyline to map
    folium.PolyLine(route_coords, color="red", weight=5).add_to(m)

# Display the map (Note: in Jupyter, interaction is limited — clicking doesn't trigger real events)
display(m)


TypeError: Nominatim did not geocode query 'Antananarivo, Madagascar' to a geometry of type (Multi)Polygon.